# NoCap-Test: Smart Filtering vs. baseline on Google Colab

Trains GPT-2 124M on FineWeb twice on the **same GPU** — the unmodified baseline and Smart Filtering (see `IDEA.md`) — and compares them.

**Runtime → Change runtime type → GPU.** Free T4: `quick` only; Colab Pro A100/L4: `full` fits.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Setup

In [ ]:
import os
USE_DRIVE = True  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/nocap'   # logs + checkpoints survive a disconnect
else:
    OUT = '/content/nocap-out'
os.makedirs(OUT, exist_ok=True)

SRC = '/content/BottleCapAI_NoCap-Test'
if not os.path.exists(SRC):
    !git clone https://github.com/vbtronic/BottleCapAI_NoCap-Test.git {SRC}
%cd {SRC}
!git pull -q
!pip install -q huggingface_hub

### 2. Configuration

| preset | tokens/step | steps | T4 | L4 | A100 | H100 |
|---|---|---|---|---|---|---|
| `quick` | 131k | 1200 | ~2.7 h | ~1 h | ~25 min | ~15 min |
| `full` | 524k | 4768, stops at val 3.3821 | ~45 h ✗ | ~16 h | ~4 h | ~2–2.5 h |

`quick` does not reach 3.3821 — the evaluation then compares how many steps each run needs to reach the
baseline's final val loss. Both runs train **one after the other on the same GPU**, the fairest comparison.

In [ ]:
PRESET = "quick"   #@param ["quick", "full"]
POOL = 1.25        #@param {type:"number"}
SCORE_LEN = 64     #@param {type:"integer"}
BATCH_SIZE = 16    #@param {type:"integer"}

import sys, importlib
sys.path.insert(0, SRC)
import experiments as ex
importlib.reload(ex)
RUN_DIR = f"{OUT}/{PRESET}"
os.makedirs(RUN_DIR, exist_ok=True)

def cmd(name, pool=None, resume=""):
    c = ex.train_cmd(PRESET, f"{RUN_DIR}/{name}", f"{RUN_DIR}/{name}/ckpt", batch_size=BATCH_SIZE,
                     pool=pool, score_len=SCORE_LEN if pool else None, extra="--save_every 200")
    if resume:
        c += f" --resume {resume}"
    return c + f" 2>&1 | tee -a {RUN_DIR}/{name}.txt"

print("shards to download:", ex.shards_needed(PRESET, POOL))

### 3. Data
Each shard is 100M tokens (~200 MB). Smart Filtering reads `POOL`× more data than it trains on.

In [ ]:
!python data/cached_fineweb10B.py {ex.shards_needed(PRESET, POOL)}
!du -sh data/fineweb10B

### 4. Baseline
After a disconnect set `RESUME` to the `latest.pt` printed in the log (`.../ckpt/<run_id>/latest.pt`).

In [ ]:
RESUME = ""  #@param {type:"string"}
!{cmd("baseline", resume=RESUME)}

### 5. Smart Filtering

In [ ]:
RESUME = ""  #@param {type:"string"}
!{cmd("smartfilter", pool=POOL, resume=RESUME)}

### 6. Compare
`time ratio` < 1.0 means Smart Filtering reaches the target faster (steps ratio × step cost).

In [ ]:
text = ex.evaluate([f"{RUN_DIR}/baseline.txt", f"{RUN_DIR}/smartfilter.txt"], plot=f"{RUN_DIR}/compare.png")
from IPython.display import Image, display
display(Image(f"{RUN_DIR}/compare.png"))